# Phase 2 — LLC Calibration (interactive)

Run **after** Phase 1 training is complete.

**Kaggle:** Add the Phase 1 notebook's output version as a dataset input
(notebook settings → Add data → your notebook → version N).
Then run cells one by one (Shift+Enter) — **do not** Save and Run All,
because you need to inspect the chain trace plot before filling in the save cell.

**Takes ≈ 20 min on T4.** No GPU strictly required but speeds up the SGLD chains.

## Section 0 — Setup

In [ ]:
import os, sys, shutil, subprocess

PLATFORM = "kaggle"   # "kaggle" or "colab"
REPO_URL  = "https://github.com/makataomu/slt-diplomka"

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    REPO_DIR    = "/content/slt"
    PERSIST_DIR = "/content/drive/MyDrive/slt_persist"
    for d in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
else:
    REPO_DIR    = "/kaggle/working/slt"
    PERSIST_DIR = None

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Pulled latest from GitHub")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned from GitHub")

if PLATFORM == "colab":
    lnk = f"{REPO_DIR}/results"
    if os.path.islink(lnk): os.unlink(lnk)
    elif os.path.isdir(lnk): shutil.rmtree(lnk)
    os.symlink(f"{PERSIST_DIR}/results", lnk)
    print(f"results/ -> {PERSIST_DIR}/results")
else:
    for sub in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{REPO_DIR}/{sub}", exist_ok=True)
    # Restore checkpoints from Phase 1 output dataset
    for inp in sorted(os.listdir("/kaggle/input")):
        prev = f"/kaggle/input/{inp}/slt/results"
        if os.path.exists(prev):
            print(f"Restoring from /kaggle/input/{inp}/ ...")
            for sub in ["checkpoints", "metrics"]:
                src, dst = f"{prev}/{sub}", f"{REPO_DIR}/results/{sub}"
                if os.path.exists(src):
                    for item in os.listdir(src):
                        s, d = f"{src}/{item}", f"{dst}/{item}"
                        if not os.path.exists(d):
                            (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
            print("  Done.")
            break

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"\nReady. Platform={PLATFORM} | cwd={os.getcwd()}")

# Verify the calibration checkpoint exists
from pathlib import Path
ckpt = Path("results/checkpoints/ratio_0.50/seed_0/epoch_06000.pt")
if ckpt.exists():
    print(f"Calibration checkpoint found: {ckpt}")
else:
    print(f"WARNING: {ckpt} not found — did you add Phase 1 output as dataset input?")


## Section 1 — Install dependencies

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
print(f"devinterp {importlib.metadata.version('devinterp')}  "
      f"| torch {torch.__version__}  "
      f"| device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


## Section 2 — Run calibration

Uses the final checkpoint of `ratio=0.50 seed=0`.
Runs 8 SGLD chains × 500 draws — prints per-chain stats when done.

In [ ]:
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate


## Section 3 — Inspect chain traces

**Good calibration:** all chains fluctuate in a stable band — no divergence, no flatline.
- Chains diverge upward → reduce `epsilon` (e.g. `1e-5`)
- Chains flatline / don't mix → increase `epsilon` or `num_draws`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

traces = np.load("results/metrics/calibration_traces.npy")
print(f"Traces shape: {traces.shape}  (chains × draws)")
for i, c in enumerate(traces):
    print(f"  Chain {i}: mean={c.mean():.4f}  std={c.std():.4f}  "
          f"min={c.min():.4f}  max={c.max():.4f}")

fig, ax = plt.subplots(figsize=(11, 4))
for i, c in enumerate(traces):
    ax.plot(c, lw=0.8, alpha=0.75, label=f"Chain {i}")
ax.set(xlabel="Draw", ylabel="Loss (SGLD)",
       title="Calibration chains — should mix in a stable band")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout(); plt.show()


## Section 4 — Save calibrated hyperparams

**Edit the values below** based on what you saw in the trace plot, then run the cell.
Start from `nbeta=46.2` (= `default_nbeta(256)`) and `epsilon=1e-4`.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

# ── EDIT THESE after inspecting traces ───────────────────────────────────────
CALIBRATED = dict(
    calibrated            = True,
    epsilon               = 1e-4,
    nbeta                 = 46.2,   # default_nbeta(256) = 256/log(256)
    gamma                 = 10.0,
    num_chains            = 8,
    num_draws             = 500,
    num_burnin_steps      = 100,
    calibration_checkpoint= "results/checkpoints/ratio_0.50/seed_0/epoch_06000.pt",
    calibration_date      = str(date.today()),
    calibration_notes     = "",  # e.g. "chains stable, no divergence"
)
# ─────────────────────────────────────────────────────────────────────────────

yaml_str = yaml.dump(CALIBRATED, default_flow_style=False)
Path("configs/llc_calibration.yaml").write_text(yaml_str)

# Back up so Phase 3 can restore it
if PLATFORM == "colab":
    import shutil
    shutil.copy("configs/llc_calibration.yaml",
                f"{PERSIST_DIR}/llc_calibration.yaml")
    print(f"Backed up to Drive")
else:  # kaggle
    import shutil
    shutil.copy("configs/llc_calibration.yaml",
                "/kaggle/working/llc_calibration.yaml")
    print("Saved to /kaggle/working/llc_calibration.yaml (appears in session output)")

print("\n" + yaml_str)
